# Create emission inventory from single trajectory

Data is pulled either from the private jet server or from adsbexchange.com 
directly.

In [ ]:
import sys
import requests
import json
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import seaborn as sns

# load local openap
sys.path.insert(1, "d:/no-backup/business-jets/openap")
import openap
print(openap.__file__)  # ensure the local version is loaded


# load json (examples)
# url = "https://private-jets.fra1.digitaloceanspaces.com/globe_history/2023-05-04/trace_full_a7a233.json"
# url = "https://private-jets.fra1.digitaloceanspaces.com/globe_history/2022-04-24/trace_full_3cd2e5.json"
# url = "https://private-jets.fra1.digitaloceanspaces.com/globe_history/2022-11-02/trace_full_a9ff1e.json"
# url = "https://private-jets.fra1.digitaloceanspaces.com/globe_history/2022-03-17/trace_full_a06e4a.json"
url = "https://samples.adsbexchange.com/traces/2022/03/01/4a/trace_full_a06e4a.json"

response = requests.get(url)
data = response.json()

# extract metadata
metadata = {k: v for k, v in data.items() if k != "trace"}
print(metadata["t"])

# convert "trace" into a dataframe
df = pd.DataFrame(data["trace"], columns=[
    "dtime", "lat", "lon", "alt_ft", "spd_kt", "track",
    "flags", "roc_fpm", "extra_data", "source", "unknown4", "unknown5",
    "unknown6", "unknown7"
])
df = df[["dtime", "lat", "lon", "alt_ft", "spd_kt", "flags", "roc_fpm"]]
df["alt_ft"] = df["alt_ft"].replace("ground", 0).astype(float)
df["press_hpa"] = openap.aero.pressure(df["alt_ft"] * openap.aero.ft) / 1e2
df["ts"] = pd.to_datetime(metadata["timestamp"] + df["dtime"], unit="s")
df["lon"] = df["lon"] % 360.0  # convert to 0 -> 360 deg

# add leg counter
leg_flags = (df["flags"] & 2).astype(bool)  # bitwise and
df["leg"] = leg_flags.cumsum()

# calculate a delta timestep whilst considering legs
df["d_leg"] = df.leg.diff().fillna(0)
df["d_ts"] = d_ts = df.ts.diff().dt.total_seconds().bfill()
df.loc[df.d_leg > 0, "d_ts"] = 0

# calculate a delta distance whilst considering legs
df["lat_prev"] = df["lat"].shift(1)
df["lon_prev"] = df["lon"].shift(1)
df["dist"] = openap.aero.distance(
    df.lat_prev, df.lon_prev, df.lat, df.lon, df.alt_ft * openap.aero.ft
) / 1e3
df.loc[df["d_leg"] > 0, "dist"] = 0
df = df.drop(columns=["d_leg", "lat_prev", "lon_prev"])

# drop NaN values
df = df.dropna(subset=["spd_kt", "alt_ft", "roc_fpm"], ignore_index=True)

df

Using OpenAP, define the trajectory and identify the phase labels.

In [ ]:
from openap.addon import bada3
from openap.phase import FlightPhase

bada_version = "LIAM"
bada_path = None
ac_model = bada3.load_bada3(metadata["t"], bada_version, bada_path)

# identify flight phases
fp = FlightPhase()
fp.set_trajectory(
    (df.ts - df.ts.iloc[0]).dt.total_seconds(),
    df["alt_ft"], df["spd_kt"], df["roc_fpm"]
)
labels = fp.phaselabel()
df["phase"] = labels

df.phase.unique()

In [ ]:
fuelflow = bada3.FuelFlow(metadata["t"], bada_version=bada_version)

# assume starting mass is reference mass
mass_current = ac_model.MREF

# map phases to fuel flow methods
phase_map = {
    "GND": None,
    "CL": fuelflow.nominal,
    "DE": fuelflow.idle,
    "LVL": fuelflow.enroute,
    "CR": fuelflow.enroute,
    "NA": None,
}

# calculate fuel flow and fuel per timestep
ff_lst = []
fuel_lst = []
for row in df.itertuples(index=False):
    ff_method = phase_map.get(row.phase)
    if row.d_ts == 0 or not ff_method:
        ff = 0.0
    else:
        ff = ff_method(
            mass=mass_current,
            tas=row.spd_kt,
            alt=row.alt_ft,
            vs=row.roc_fpm,
        )[0][0]
    fuel = ff * row.d_ts
    mass_current -= fuel
    ff_lst.append(ff)
    fuel_lst.append(fuel)

# assign to dataframe
df = df.assign(fuel_flow=ff_lst, fuel=fuel_lst)
df

In [ ]:
# plot data
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.set_extent([-180.0, 180.0, -90.0, 90.0])

plt.plot(df.lon, df.lat, "r", transform=ccrs.Geodetic())

In [ ]:
# show trajectories

phasecolors = {
    "GND": "black",
    "CL": "green",
    "DE": "blue",
    "LVL": "cyan",
    "CR": "purple",
    "NA": "red",
}

colors = [phasecolors[lbl] for lbl in labels]

fig, axes = plt.subplots(nrows=4, sharex="all")

axes[0].scatter(df.ts, df.alt_ft, marker=".", c=colors, lw=0)
axes[0].set_ylabel("altitude (ft)")

axes[1].scatter(df.ts, df.spd_kt, marker=".", c=colors, lw=0)
axes[1].set_ylabel("speed (kt)")

axes[2].scatter(df.ts, df.roc_fpm, marker=".", c=colors, lw=0)
axes[2].set_ylabel("roc (fpm)")

axes[3].scatter(df.ts, df.fuel_flow, marker=".", c=colors, lw=0)
axes[3].set_ylabel("fuel flow (kg/s)")

In [ ]:
# compare to simple gph calculation
gph_dict = pd.read_csv("../data/external/aircraft_gph.csv", names=["icao", "gph"], index_col=0).to_dict()["gph"]

dff = df.query('phase != "GND" & phase != "NA"')
duration1 = dff.d_ts.sum() / 3600.0

# duration 2 is last-first for each leg
leg_time = dff.groupby("leg")["ts"].agg(["first", "last"])
leg_time["dur_s"] = (leg_time["last"] - leg_time["first"]).dt.total_seconds()
duration2 = leg_time["dur_s"].sum() / 3600.0

print(duration1, duration2)

gph_fuel = duration2 * gph_dict[metadata["t"]]
gph_kg_fuel = gph_fuel * 3.78541 * 0.8

print(f"BADA3 total fuel: {df.fuel.sum()} kg")
print(f"gph total fuel: {gph_kg_fuel} kg")

Regrid onto the OpenAirClim contrail grid. It can be any grid ultimately.

In [1]:
import xarray as xr

ds_cont = xr.load_dataset("../oac/repository/resp_cont_lf.nc")
inv_plev = ds_cont.plev.data
inv_lon = ds_cont.lon.data
inv_lat = ds_cont.lat.data

In [ ]:
lat_idxs = np.abs(inv_lat[:, np.newaxis] - df.lat.to_numpy()).argmin(axis=0)
lon_idxs = np.abs(inv_lon[:, np.newaxis] - df.lon.to_numpy()).argmin(axis=0)
plev_idxs = np.abs(inv_plev[:, np.newaxis] - df.press_hpa.to_numpy()).argmin(axis=0)

In [ ]:
fig, axes = plt.subplots(nrows=3, sharex="all")
axes[0].scatter(df.ts, inv_lat[lat_idxs])
axes[1].scatter(df.ts, inv_lon[lon_idxs])
axes[2].scatter(df.ts, inv_plev[plev_idxs])

In [ ]:
fig, axes = plt.subplots(nrows=3, sharex="all")
axes[0].scatter(df.ts, df.lat)
axes[1].scatter(df.ts, df.lon)
axes[2].scatter(df.ts, df.press_hpa)

In [ ]:
sum_fuel = np.zeros((len(inv_plev), len(inv_lat), len(inv_lon)))
sum_dist = np.zeros((len(inv_plev), len(inv_lat), len(inv_lon)))
np.add.at(sum_fuel, (plev_idxs, lat_idxs, lon_idxs), df.fuel.to_numpy())
np.add.at(sum_dist, (plev_idxs, lat_idxs, lon_idxs), df.dist.to_numpy())

In [ ]:
ds_inv = xr.Dataset(
    {"fuel": (("plev", "lat", "lon"), sum_fuel)},
    coords={"plev": ds_cont.plev, "lat": ds_cont.lat, "lon": ds_cont.lon}
)
ds_inv.fuel.sel(plev=1000).plot()